# Citi Bike: simple forecasting baselines

**Contribution owner: Tomislav Novosel.** Implementation, experiments and initial explanations were prepared with AI assistance on 9 October 2026. Human review and substantive changes should be recorded when they happen; see [the assistance record](../docs/AI_USE.md).

These are development experiments, not final-test results. The training/tuning dates end in June; July-September is development validation. October-December has not been scored.

The searches were executed with the helper scripts and their actual reports are saved. By default this notebook reviews those reports quickly. Set `CITIBIKE_RETRAIN=1` or use the runner's `--train` option to execute the search again. The XGBoost search needs a CUDA GPU; viewing results and running the saved app do not.


A useful model must beat simple guesses. We compare a constant training mean, yesterday, last week, a recent mean and a stronger four-week seasonal average.

In [3]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/citibike_forecast.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
from citibike_forecast import REPORTS, MODELS, development_data, chronological_cv
RUN_TRAINING = os.environ.get('CITIBIKE_RETRAIN') == '1'
def report(name):
    return json.loads((REPORTS / (name + '_run.json')).read_text(encoding='utf8'))
print('Mode:', 'execute training' if RUN_TRAINING else 'review actual saved experiments')
from citibike_forecast import run_baselines
from citibike_additional import run_search
if RUN_TRAINING:
    run_baselines()
    run_search('seasonal_average')
names = ['training_mean','lag_1','lag_7','mean_7','ridge_reference','seasonal_average_tuned']
display(pd.DataFrame([{'model': n, **report(n)['validation']} for n in names]).sort_values('mae'))


Mode: review actual saved experiments


,model,mae,rmse,wape,bias,days
5,seasonal_average_tuned,16188.702378,23247.761941,0.134234,1360.381363,92
3,mean_7,17716.462733,24812.687529,0.146902,1366.440994,92
4,ridge_reference,17850.334487,23119.476403,0.148012,-6392.475609,92
1,lag_1,17914.978261,27065.592069,0.148548,114.195652,92
2,lag_7,18476.902174,28229.779259,0.153208,2869.097826,92
0,training_mean,37173.449986,40428.708972,0.308237,-31344.868429,92


All guesses predict the same 92 validation dates. The four-week baseline has nine configurations; its choice was made on training CV, not these validation answers.

In [4]:
display(pd.Series(report('seasonal_average_tuned')['search']))
print('MAE is the average size of a daily mistake in rides. Lower is better.')
print('WAPE adds all absolute mistakes and divides by all actual rides; it is not accuracy.')


MAE is the average size of a daily mistake in rides. Lower is better.
WAPE adds all absolute mistakes and divides by all actual rides; it is not accuracy.


method                                             GridSearchCV
configurations                                                9
cv_fits                                                      27
best_parameters               {'adjustment': 1.0, 'decay': 1.0}
best_cv_mae                                        14545.443801
elapsed_seconds                                        3.302988
validation_used_for_search                                False
dtype: object